<a href="https://colab.research.google.com/github/luiza-vargas/onl/blob/main/C%C3%B3pia_de_projeto_ONl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Dados
#---------------------------------------------------------------------------------------------------------------
#Struct auxiliar para armazenar os dados que serão devolvidos pela função retornaDados
#---------------------------------------------------------------------------------------------------------------
mutable struct containerDados
    demanda::Vector #Demanda para cada mês
    NHE::Number #Número de hidrelétricas
    NTE::Number #Número de termelétricas
    NT::Number #Número de meses no período considerado
    term_pot::Vector #Potencia máxima de cada térmica
    NUsinasAntes::Vector #Número de usinas a montante de cada uma das hidrelétricas
    QUsinasAntes::Matrix #Vetor indicando o indice das usinas a montante de uma dada usina i
    vaz_increm::Matrix #Matriz de vazoes incrementais de uma usina i em um mês j
    hidr_vol_inicial::Vector #Volume inicial de cada hidrelétrica
    cvu::Vector #Custo variável unitário das térmicas
    cust_def::Number #Custo do déficit
    hidr_pot::Vector #potencia maxima de cada hidreletrica
    hidr_prod_esp::Vector #produtibilidade específica de cada hidrelétrica
    hidr_prod_media::Vector #Produtibilidade média de cada hidrelétrica
    hidr_turb_max::Vector #máximo que pode ser turbinado por cada hidrelétrica
    hidr_perd_hid::Vector #Perda hidráulica
    hidr_vol_max::Vector #Volume máximo de cada hidrelétrica
    coef_vol::Matrix #coeficientes do polinomio cota volume
    coef_vaz::Matrix #coeficientes do polinomio cota vazão
    vol_min::Vector #Volume minimo de cada hidrelétrica
    deltaT::Vector #Duração de cada período no horizonte estipulado
    function containerDados(demanda,NHE,NTE,NT,term_pot,NUsinasAntes,QUsinasAntes,vaz_increm,hidr_vol_inicial,cvu,cust_def,hidr_pot,hidr_prod_esp,hidr_prod_media,hidr_turb_max,hidr_perd_hid,hidr_vol_max,coef_vol,coef_vaz,vol_min,deltaT)
        return new(demanda,NHE,NTE,NT,term_pot,NUsinasAntes,QUsinasAntes,vaz_increm,hidr_vol_inicial,cvu,cust_def,hidr_pot,hidr_prod_esp,hidr_prod_media,hidr_turb_max,hidr_perd_hid,hidr_vol_max,coef_vol,coef_vaz,vol_min,deltaT)
    end
end

#---------------------------------------------------------------------------------------------------------------
#Função que retorna os dados presentes nos slides do CEPEL, que serão utilizados na construção e resolução dos modelos.
#Recebe..: Nada
#Retorna.: Dados contidos nos slides, como o vetor de demanda para cada mês e o custo de produção de cada usina...
#---------------------------------------------------------------------------------------------------------------

function retornaDados()
    #Número de usinas imediatamente a montante de uma dada usina hidrelétrica i, seguindo a ordem estabelecida nos slides
    #i=1->Retiro baixo, 2->três marias, 3->queimado, 4-> sobradinho, 5 - itaparica, 6-Comp. paf-mox, 7-xingo
    NUsinasAntes=[0;1;0;2;1;1;1]

    #Quais usinas se localizam à montante de uma usina i (seguindo a mesma ordem mencionada antes de NUsinas_Antes)
    QUsinasAntes=transpose([0 0; 1 0; 0 0; 2 3; 4 0; 5 0; 6 0])

    #Número de térmicas(NTE), hidrelétricas(NHE) e meses(NT)
    NHE=7
    NTE=8
    NT=24

    #demanda de energia elétrica em MWmedio em cada mes
    demanda=[10507.0,9736.0,9491.0,8745.0,6876.0,5858.0,5773.0,6563.0,8428.0,9433.0,9891.0,10698.0,10507.0,9736.0,9491.0,8745.0,6876.0,5858.0,5773.0,6563.0,8428.0,9433.0,9891.0,10698.0]

    #Potência máxima de cada termelétrica
    term_pot = [136.00,87.00,140.00,929.00,437.00,54.00,169.00,594.00]

    #Potência total de todas as termelétricas -> É a soma de todos os term_pot[i]
    #term_tot_pot = 2546.00

    #Custo variável unitário de produção para cada termelétrica
    cvu = [1927.00,523.00,1493.00,880.00,1257.00,2632.00,1257.00,2014.00]

    #Custo de cada MW não produzido - déficit de produção
    cust_def=7643.82

    #Especificações das hidrelétricas

    #Potência e potência total
    hidr_pot = [82.74,362.89,100.65,1008.88,1422.39,3883.02,3079.08]
    #hidr_tot_pot=9939.65

    #Volume máximo em hm^3
    hidr_vol_max = [40.87,15278.00,461.75,28669.00,3548.00,0.00,0.00]

    #Produtibilidade específica para cada hidrelétrica
    hidr_prod_esp = [0.00340684, 0.00331445, 0.00338821, 0.00337681, 0.00332281, 0.00343536, 0.00342738]

    #Produtibilidade média
    hidr_prod_media = [0.127224335, 0.165779468, 0.623726236, 0.093117871, 0.168250951, 0.388326996, 0.409923954]

    #Volume máximo de água, em hm^3, que pode ser turbinada
    hidr_turb_max = [662.20, 2112.89, 166.44, 10690.02, 8263.97, 9999.36, 7511.35]

    #Perda hidraulica
    hidr_perd_hid = [0.6, 0.6, 2.5, 0.2, 0.5, 0.6, 0.8]

    #Volume inicial de cada usina
    hidr_vol_inicial = [27.38, 7654.28, 258.12, 14821.87, 2117.09, 0.00, 0.00]

    #Vazoes afluentes incrementais = chuvas
    #Linhas, por número, representam nessa ordem:
    #1->Retiro baixo, 2->três marias, 3->queimado, 4-> sobradinho, 5 - itaparica, 6-Comp. paf-mox, 7-xingo
    #primeiras 12 colunas referentes aos meses de 2021, as 12 seguintes se referem aos meses de 2022
    vaz_increm=[
        2593.18	736.40	844.23	570.71	415.54	386.61	278.78	228.81	263.00	247.22	299.82	881.05	765.33	817.93	441.84	305.08	241.96	286.67	178.84	178.84	110.46	189.36	397.13	402.39;
        6619.71	2348.59	2772.02	1917.27	1096.71	1091.45	783.74	712.73	405.02	641.72	825.82	3424.26	2495.87	2445.90	1583.26	1038.85	959.95	710.10	494.44	473.40	405.02	499.70	1393.90	1433.35;
        136.76	105.20	152.54	194.62	136.76	110.46	89.42	73.64	63.12	60.49	81.53	123.61	99.94	147.28	110.46	86.79	71.01	60.49	49.97	39.45	31.56	44.71	142.02	191.99;
        6961.61	8965.67	7963.64	7976.79	4762.93	3006.09	2251.28	2132.93	2022.47	1817.33	2324.92	4634.06	5430.95	5425.69	6006.92	2608.96	1677.94	1441.24	1480.69	1283.44	1083.56	1117.75	3311.17	6496.10;
        0.00	1236.10	0.00	1073.04	378.72	97.31	178.84	60.49	18.41	26.30	0.00	0.00	468.14	0.00	497.07	263.00	273.52	63.12	2.63	89.42	105.20	0.00	0.00	0.00;
        0.00	391.87	0.00	397.13	328.75	52.60	84.16	105.20	155.17	236.70	47.34	0.00	139.39	10.52	147.28	131.50	68.38	23.67	31.56	5.26	155.17	65.75	0.00	0.00;
        0.00	0.00	0.00	0.00	2.63	0.00	0.00	0.00	0.00	2.63	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	2.63	2.63	0.00
    ]

    #Coeficientes do polinomio cota-volume para hidrelétrica
    #Linhas, por número, representam nessa ordem:
    #1->Retiro baixo, 2->três marias, 3->queimado, 4-> sobradinho, 5 - itaparica, 6-Comp. paf-mox, 7-xingo
    #Coluna i representam o coeficiente dos termos de grau i-1 do polinomio
    #Exemplo: A primeira linha inteira nos dá os coeficientes do polinomio cota volume da usina de Retiro Baixo
    coef_vol=[
        5.92E+02	1.64E-1	-2.61E-4	0.00000E+00	0.00000E+00;
        5.30E+02	6.08E-3	-4.84E-07	2.20E-11	-3.85E-16;
        8.02E+02	1.14E-1	-1.98E-4	1.44E-7	-2.49E-17;
        3.74E+02	1.40E-3	-5.35E-8	1.16E-12	-9.55E-18;
        2.76E+02	6.76E-3	-8.87E-07	7.07E-11	-2.24E-15;
        2.51E+02	0.00000E+00	0.00000E+00	0.00000E+00	0.00000E+00;
        1.37E+02	0.00000E+00	0.00000E+00	0.00000E+00	0.00000E+00
    ]

    #Coeficientes do polinomio cota-vazão para hidrelétrica
    #Linhas, por número, representam nessa ordem:
    #1->Retiro baixo, 2->três marias, 3->queimado, 4-> sobradinho, 5 - itaparica, 6-Comp. paf-mox, 7-xingo
    #Coluna i representam o coeficiente dos termos de grau i-1 do polinomio
    #Exemplo: A primeira linha inteira nos dá os coeficientes do polinomio cota vazão da usina de Retiro Baixo
    coef_vaz=[
        5.770000E+02	6.877187E-03	-2.099505E-06	3.629827E-10	-2.358950E-14;
        5.146558E+02	1.606860E-03	-2.552750E-07	2.885479E-11	-1.179780E-15;
        6.362000E+02	3.073000E-02	-1.553100E-04	5.084740E-07	-6.098659E-10;
        3.596538E+02	1.964010E-03	-2.968730E-07	2.508280E-11	-7.702299E-16;
        2.515000E+02	0.000000E+00	0.000000E+00	0.000000E+00	0.000000E+00;
        1.380000E+02	0.000000E+00	0.000000E+00	0.000000E+00	0.000000E+00;
        1.003861E+01	6.665461E-03	-2.454770E-06	4.559482E-10	-3.144850E-14
    ]

    #Volume minimo de cada hidrelétrica em hm^3
    #Vamos considerar inicialmente como sendo 0, mas podemos alterar futuramente para gerar mais casos de teste
    # vol_min=[0.0, 0, 0, 0, 0, 0, 0]
    vol_min=hidr_vol_max * 0.1

    #duração de cada período - Considerando os anos de 2021 e 2022 - colocamos a duração de cada mes em horas
    deltaT=[744,672,744,720,744,720,744,744,720,744,720,744,744,672,744,720,744,720,744,744,720,744,720,744.0]

    #Colocando as informações em um struct e retornando
    #resp=containerDados(demanda,NHE,NTE,NT,term_pot,NUsinasAntes,QUsinasAntes,vaz_increm,hidr_vol_inicial,cvu,cust_def,hidr_pot,hidr_prod_esp,hidr_prod_media,hidr_turb_max,hidr_perd_hid,hidr_vol_max,coef_vol,coef_vaz,vol_min)
    resp=containerDados(demanda,NHE,NTE,NT,term_pot,NUsinasAntes,QUsinasAntes,vaz_increm,hidr_vol_inicial,cvu,cust_def,hidr_pot,hidr_prod_esp,hidr_prod_media,hidr_turb_max,hidr_perd_hid,hidr_vol_max,coef_vol,coef_vaz,vol_min,deltaT)
    return resp
end

retornaDados (generic function with 1 method)

In [ ]:
#Arquivo JuliaLinear
import Pkg; Pkg.add("JuMP")
import Pkg; Pkg.add("Gurobi")

# ENV["GRB_LICENSE_FILE"] = "gurobi.lic"


using JuMP
using Gurobi
using LinearAlgebra

function resolvePL_com_verificacao()
    dados=retornaDados()

    # Redefinindo algumas variáveis para simplificar a escrita do código
    NHE=dados.NHE; NTE=dados.NTE; NT=dados.NT; vol_min=dados.vol_min;
    hidr_pot=dados.hidr_pot; term_pot=dados.term_pot; hidr_turb_max=dados.hidr_turb_max;
    demanda=dados.demanda; prod_media=dados.hidr_prod_media; vol_inicial=dados.hidr_vol_inicial;
    vaz_increm=dados.vaz_increm; NUsinasAntes=dados.NUsinasAntes; QUsinasAntes=dados.QUsinasAntes;
    deltaT=dados.deltaT; cvu=dados.cvu; cust_def=dados.cust_def; hidr_vol_max=dados.hidr_vol_max;

    # Declarando o modelo
    linModel=Model(Gurobi.Optimizer)

    # Variáveis
    @variable(linModel, 0<=gh[i=1:NHE,j=1:NT]<=hidr_pot[i])
    @variable(linModel, vol_min[i]<=vol[i=1:NHE,j=1:NT]<=hidr_vol_max[i])
    @variable(linModel, 0<=turb[i=1:NHE,j=1:NT]<=hidr_turb_max[i])
    @variable(linModel, 0<=vert[i=1:NHE,j=1:NT])
    @variable(linModel, 0<=gt[i=1:NTE,j=1:NT]<=term_pot[i])
    @variable(linModel,0<=deficit[j=1:NT])

    # Restrições
    @constraint(linModel,atendimentoDemanda[j=1:NT], sum(gh[i,j] for i in 1:NHE)+sum(gt[i,j] for i in 1:NTE)+deficit[j]==demanda[j])
    @constraint(linModel,prodMedia[i=1:NHE,j=1:NT], gh[i,j]==prod_media[i]*turb[i,j])
    @constraint(linModel,balanHidrInicial[i=1:NHE], vol[i,1]-vol_inicial[i]+turb[i,1]+vert[i,1]-sum(turb[QUsinasAntes[k,i],1]+vert[QUsinasAntes[k,i],1] for k in 1:NUsinasAntes[i])==vaz_increm[i,1])
    @constraint(linModel,balanHidr[i=1:NHE,j=2:NT], vol[i,j]-vol[i,j-1]+turb[i,j]+vert[i,j]-sum(turb[QUsinasAntes[k,i],j]+vert[QUsinasAntes[k,i],j] for k in 1:NUsinasAntes[i])==vaz_increm[i,j])

    # Função Objetivo
    @objective(linModel, Min, sum(deltaT[j] * (sum(cvu[i]*gt[i,j] for i in 1:NTE) + cust_def*deficit[j]) for j in 1:NT))

    println("Otimizando o modelo linear...")
    optimize!(linModel)

    return linModel, dados
end


modelo_resolvido, dados = resolvePL_com_verificacao()

println("\n\n===============================================")
println("--- INICIANDO VERIFICAÇÃO DAS RESTRIÇÕES ---")
println("===============================================")

if has_values(modelo_resolvido)
    sol_gh = value.(modelo_resolvido[:gh])
    sol_gt = value.(modelo_resolvido[:gt])
    sol_deficit = value.(modelo_resolvido[:deficit])
    sol_vol = value.(modelo_resolvido[:vol])
    sol_turb = value.(modelo_resolvido[:turb])
    sol_vert = value.(modelo_resolvido[:vert])

    # Tolerância para comparações de ponto flutuante
    TOLERANCIA = 1e-5

    # 1. Verificando a Restrição de Atendimento à Demanda
    println("\n[1] Verificando Atendimento à Demanda: sum(gh) + sum(gt) + deficit == demanda")
    for j in 1:dados.NT
        lhs_demanda = sum(sol_gh[:, j]) + sum(sol_gt[:, j]) + sol_deficit[j]
        rhs_demanda = dados.demanda[j]
        diferenca = abs(lhs_demanda - rhs_demanda)

        status = diferenca < TOLERANCIA ? "OK" : "FALHOU"
        println("   Período ", j, ": LHS = ", round(lhs_demanda, digits=4),
                  ", RHS = ", round(rhs_demanda, digits=4),
                  ", Diferença = ", round(diferenca, digits=8), " -> [", status, "]")
    end

    # 2. Verificando a Restrição de Balanço Hídrico
    println("\n[2] Verificando Balanço Hídrico: vol[t] - vol[t-1] + turb[t] + vert[t] - ... == vaz_inc[t]")
    for i in 1:dados.NHE
        # Período 1 (inicial)
        j = 1
        afluencia_montante = sum(sol_turb[dados.QUsinasAntes[k,i], j] + sol_vert[dados.QUsinasAntes[k,i], j] for k in 1:dados.NUsinasAntes[i]; init=0)
        lhs_balanco = sol_vol[i,j] - dados.hidr_vol_inicial[i] + sol_turb[i,j] + sol_vert[i,j] - afluencia_montante
        rhs_balanco = dados.vaz_increm[i,j]
        diferenca = abs(lhs_balanco - rhs_balanco)
        status = diferenca < TOLERANCIA ? "OK" : "FALHOU"
        println("   Usina ", i, ", Período ", j, ": Diferença = ", round(diferenca, digits=8), " -> [", status, "]")

        # Períodos 2 em diante
        for j in 2:dados.NT
            afluencia_montante = sum(sol_turb[dados.QUsinasAntes[k,i], j] + sol_vert[dados.QUsinasAntes[k,i], j] for k in 1:dados.NUsinasAntes[i]; init=0)
            lhs_balanco = sol_vol[i,j] - sol_vol[i,j-1] + sol_turb[i,j] + sol_vert[i,j] - afluencia_montante
            rhs_balanco = dados.vaz_increm[i,j]
            diferenca = abs(lhs_balanco - rhs_balanco)
            status = diferenca < TOLERANCIA ? "OK" : "FALHOU"
            println("   Usina ", i, ", Período ", j, ": Diferença = ", round(diferenca, digits=8), " -> [", status, "]")
        end
    end

else
    println("Modelo não possui uma solução válida para ser verificada.")
end

In [ ]:
#Arquivo JuliaNLinear
import Pkg;
Pkg.add("JuMP")
Pkg.add("Ipopt")
using JuMP
using Ipopt
using LinearAlgebra

# Arquivo de dados
include("dados.jl")

function resolvePNL()
    # Recebendo os dados
    dados=retornaDados()

    NHE=dados.NHE
    NTE=dados.NTE
    NT=dados.NT
    vol_min=dados.vol_min
    hidr_pot=dados.hidr_pot
    term_pot=dados.term_pot
    hidr_turb_max=dados.hidr_turb_max
    demanda=dados.demanda
    vol_inicial=dados.hidr_vol_inicial
    vaz_increm=dados.vaz_increm
    NUsinasAntes=dados.NUsinasAntes
    QUsinasAntes=dados.QUsinasAntes
    deltaT=dados.deltaT
    cvu=dados.cvu
    cust_def=dados.cust_def
    hidr_vol_max=dados.hidr_vol_max
    hidr_perd_hid=dados.hidr_perd_hid
    hidr_prod_esp=dados.hidr_prod_esp
    coef_vol=dados.coef_vol
    coef_vaz=dados.coef_vaz

    # Construindo o modelo e definindo o solver Ipopt
    NlinModel = Model(Ipopt.Optimizer)
    set_attribute(NlinModel, "max_cpu_time", 3600.0) # Limite de tempo de execução
    set_attribute(NlinModel, "print_level", 5)      # Nível de print do solver (5 é um bom padrão)

    # --- Variáveis do modelo ---
    @variable(NlinModel, 0 <= gh[i=1:NHE, j=1:NT] <= hidr_pot[i])
    @variable(NlinModel, 0 <= gt[i=1:NTE, j=1:NT] <= term_pot[i])
    @variable(NlinModel, vol_min[i] <= vol[i=1:NHE, j=1:NT] <= hidr_vol_max[i])
    @variable(NlinModel, 0 <= vert[i=1:NHE, j=1:NT])
    @variable(NlinModel, 0 <= turb[i=1:NHE, j=1:NT] <= hidr_turb_max[i])
    @variable(NlinModel, 0 <= deficit[j=1:NT])

    # Variáveis auxiliares para o cálculo não linear
    @variable(NlinModel, alturalq[i=1:NHE, j=1:NT] >= 0) # Altura de queda líquida
    @variable(NlinModel, altMon[i=1:NHE, j=1:NT] >= 0)   # Altura a montante
    @variable(NlinModel, altJus[i=1:NHE, j=1:NT] >= 0)   # Altura a jusante
    @variable(NlinModel, volTot[i=1:NHE, j=1:NT] >= 0)   # Volume médio no período
    @variable(NlinModel, defTot[i=1:NHE, j=1:NT] >= 0)   # Defluência total
    @variable(NlinModel, prodAltExata[i=1:NHE, j=1:NT] >= 0) # Produtibilidade na altura exata

    # --- Restrições do modelo ---

    # 1- Atendimento da demanda (igual ao modelo linear)
    @constraint(NlinModel, atendimentoDemanda[j=1:NT], sum(gh[i,j] for i in 1:NHE) + sum(gt[i,j] for i in 1:NTE) + deficit[j] == demanda[j])

    # 2- Balanço hídrico (igual ao modelo linear)
    # Para o primeiro mês (j=1)
    @constraint(NlinModel, balanHidrInicial[i=1:NHE], vol[i,1] - vol_inicial[i] + turb[i,1] + vert[i,1] - sum(turb[QUsinasAntes[k,i],1] + vert[QUsinasAntes[k,i],1] for k in 1:NUsinasAntes[i]) == vaz_increm[i,1])
    # Para os demais meses (j>1)
    @constraint(NlinModel, balanHidr[i=1:NHE, j=2:NT], vol[i,j] - vol[i,j-1] + turb[i,j] + vert[i,j] - sum(turb[QUsinasAntes[k,i],j] + vert[QUsinasAntes[k,i],j] for k in 1:NUsinasAntes[i]) == vaz_increm[i,j])

    # 3- Definições da parte não linear da hidráulica

    # Eq. (10) Altura líquida de queda
    @constraint(NlinModel, altLiqRest[i=1:NHE, j=1:NT], alturalq[i,j] == altMon[i,j] - altJus[i,j] - hidr_perd_hid[i])

    # Eq. (12) Volume médio no período
    @constraint(NlinModel, volTotRestInicial[i=1:NHE], volTot[i,1] == (vol[i,1] + vol_inicial[i]) / 2)
    @constraint(NlinModel, volTotRest[i=1:NHE, j=2:NT], volTot[i,j] == (vol[i,j] + vol[i,j-1]) / 2)

    # Eq. (14) Defluência total
    @constraint(NlinModel, defRest[i=1:NHE, j=1:NT], defTot[i,j] == turb[i,j] + vert[i,j])

    # Eq. (11) Polinômio Cota-Volume (Altura Montante)
    @NLconstraint(NlinModel, polCotVol[i=1:NHE, j=1:NT], altMon[i,j] == coef_vol[i,1] + coef_vol[i,2]*volTot[i,j] + coef_vol[i,3]*volTot[i,j]^2 + coef_vol[i,4]*volTot[i,j]^3 + coef_vol[i,5]*volTot[i,j]^4)

    # Eq. (13) Polinômio Cota-Vazão (Altura Jusante)
    @NLconstraint(NlinModel, polCotVaz[i=1:NHE, j=1:NT], altJus[i,j] == coef_vaz[i,1] + coef_vaz[i,2]*defTot[i,j] + coef_vaz[i,3]*defTot[i,j]^2 + coef_vaz[i,4]*defTot[i,j]^3 + coef_vaz[i,5]*defTot[i,j]^4)

    # 4- Conexão da hidráulica com a geração de energia (equações CHAVE do modelo não linear)

    # Eq. (15) Produtibilidade específica
    @constraint(NlinModel, prodEspefRest[i=1:NHE, j=1:NT], prodAltExata[i,j] == hidr_prod_esp[i] * alturalq[i,j])

    # Eq. (3.1) Geração da hidrelétrica
    @NLconstraint(NlinModel, gerHidrRest[i=1:NHE, j=1:NT], gh[i,j] == prodAltExata[i,j] * turb[i,j])

    # --- Função Objetivo ---
    @objective(NlinModel, Min, sum(deltaT[j] * (sum(cvu[i] * gt[i,j] for i in 1:NTE) + cust_def * deficit[j]) for j in 1:NT))

    println("Otimizando o modelo não linear com Ipopt...")
    optimize!(NlinModel)

    println("\nStatus da Solução: ", termination_status(NlinModel))
    if has_values(NlinModel)
        println("Custo Total Ótimo: ", objective_value(NlinModel))
    end
    return NlinModel, dados
end

modelo_resolvido, dados = resolvePNL();
println("\n--- Resumo da Solução ---")
println(solution_summary(modelo_resolvido, verbose=false))

println("\n\n============================================================")
println("--- INICIANDO VERIFICAÇÃO DAS RESTRIÇÕES (NÃO LINEAR) ---")
println("============================================================")

if has_values(modelo_resolvido)
    sol_gh = value.(modelo_resolvido[:gh]); sol_gt = value.(modelo_resolvido[:gt]);
    sol_deficit = value.(modelo_resolvido[:deficit]); sol_vol = value.(modelo_resolvido[:vol]);
    sol_turb = value.(modelo_resolvido[:turb]); sol_vert = value.(modelo_resolvido[:vert]);
    sol_alturalq = value.(modelo_resolvido[:alturalq]); sol_altMon = value.(modelo_resolvido[:altMon]);
    sol_altJus = value.(modelo_resolvido[:altJus]); sol_volTot = value.(modelo_resolvido[:volTot]);
    sol_defTot = value.(modelo_resolvido[:defTot]); sol_prodAltExata = value.(modelo_resolvido[:prodAltExata]);

    # Tolerância
    TOLERANCIA = 1e-5

    # 1. Verificando a Geração Hidrelétrica (NÃO LINEAR)
    println("\n[1] Verificando Geração Hidrelétrica: gh == prodAltExata * turb")
    for j in 1:dados.NT, i in 1:dados.NHE
        lhs = sol_gh[i, j]
        rhs = sol_prodAltExata[i, j] * sol_turb[i, j]
        diferenca = abs(lhs - rhs)
        if diferenca > TOLERANCIA
            println("   Usina H ", i, ", Período ", j, ": Diferença = ", round(diferenca, digits=8), " -> [FALHOU]")
        end
    end
    println("   -> Verificação da Geração Hidrelétrica concluída.")

    # 2. Verificando o Polinômio Cota-Volume (NÃO LINEAR)
    println("\n[2] Verificando Polinômio Cota-Volume: altMon == f(volTot)")
    for j in 1:dados.NT, i in 1:dados.NHE
        lhs = sol_altMon[i, j]
        v = sol_volTot[i, j]
        # Recalcula o polinômio manualmente
        rhs = dados.coef_vol[i,1] + dados.coef_vol[i,2]*v + dados.coef_vol[i,3]*v^2 + dados.coef_vol[i,4]*v^3 + dados.coef_vol[i,5]*v^4
        diferenca = abs(lhs - rhs)
        if diferenca > TOLERANCIA
            println("   Usina H ", i, ", Período ", j, ": Diferença = ", round(diferenca, digits=8), " -> [FALHOU]")
        end
    end
    println("   -> Verificação do Polinômio Cota-Volume concluída.")

    # 3. Verificando o Balanço Hídrico (LINEAR)
    println("\n[3] Verificando Balanço Hídrico")
    for i in 1:dados.NHE
        # Período 1
        j = 1
        afluencia_montante = sum(sol_turb[dados.QUsinasAntes[k,i], j] + sol_vert[dados.QUsinasAntes[k,i], j] for k in 1:dados.NUsinasAntes[i]; init=0)
        lhs = sol_vol[i,j] - dados.hidr_vol_inicial[i] + sol_turb[i,j] + sol_vert[i,j] - afluencia_montante
        rhs = dados.vaz_increm[i,j]
        diferenca = abs(lhs - rhs)
        if diferenca > TOLERANCIA
           println("   Usina ", i, ", Período ", j, ": Diferença = ", round(diferenca, digits=8), " -> [FALHOU]")
        end

        # Períodos 2 em diante
        for j in 2:dados.NT
            afluencia_montante = sum(sol_turb[dados.QUsinasAntes[k,i], j] + sol_vert[dados.QUsinasAntes[k,i], j] for k in 1:dados.NUsinasAntes[i]; init=0)
            lhs = sol_vol[i,j] - sol_vol[i,j-1] + sol_turb[i,j] + sol_vert[i,j] - afluencia_montante
            rhs = dados.vaz_increm[i,j]
            diferenca = abs(lhs - rhs)
            if diferenca > TOLERANCIA
                println("   Usina ", i, ", Período ", j, ": Diferença = ", round(diferenca, digits=8), " -> [FALHOU]")
            end
        end
    end
    println("   -> Verificação do Balanço Hídrico concluída.")

    println("\n--- VERIFICAÇÃO CONCLUÍDA ---")
    println("(Se nenhuma mensagem de 'FALHOU' apareceu, todas as restrições verificadas foram satisfeitas)")

else
    println("Modelo não possui uma solução válida para ser verificada.")
end

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
   Installed METIS_jll ────── v5.1.3+0
   Installed Ipopt_jll ────── v300.1400.1901+0
   Installed MUMPS_seq_jll ── v500.800.200+0
   Installed ASL_jll ──────── v0.1.3+0
   Installed Hwloc_jll ────── v2.13.0+1
   Installed OpenBLAS32_jll ─ v0.3.33+1
   Installed XML2_jll ─────── v2.13.9+0
   Installed SPRAL_jll ────── v2025.9.18+0
   Installed Ipopt ────────── v1.14.3
  Installing 8 artifacts
   Installed artifact ASL                      276.3 KiB
   Installed artifact METIS                    1.1 MiB
   Installed artifact Ipopt                    1.3 MiB
   Installed artifact SPRAL                    691.9 KiB
   Installed artifact XML2                     2.5 MiB
   Installed artifact Hwloc                    3.5 MiB
   Installed arti

LoadError: SystemError: opening file "/content/dados.jl": No such file or directory